In [ ]:
# 1. Mount Google Drive.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Central path configuration and fresh extraction of the exact codebase archive.
import shutil
from pathlib import Path
from zipfile import ZipFile

DRIVE_DIR = Path("/content/drive/MyDrive/[ICLR] Embedding KD")
ARCHIVE_PATH = DRIVE_DIR / "ICLR-MDD-nqd_claude_rcm.zip"
EXTRACT_DIR = Path("/content/ICLR-MDD-nqd_claude_rcm_workspace")
STUDENT_MODEL_NAME = "google-bert/bert-base-uncased"
TEACHER_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"

assert DRIVE_DIR.is_dir(), f"Google Drive directory not found: {DRIVE_DIR}"
assert ARCHIVE_PATH.is_file(), f"Codebase archive not found: {ARCHIVE_PATH}"
assert EXTRACT_DIR == Path("/content/ICLR-MDD-nqd_claude_rcm_workspace")

# Always start from the code in the selected ZIP, never a stale extracted repo.
if EXTRACT_DIR.is_symlink():
    EXTRACT_DIR.unlink()
elif EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=False)

extract_root = EXTRACT_DIR.resolve()
with ZipFile(ARCHIVE_PATH) as archive:
    unsafe_members = []
    for member in archive.infolist():
        destination = (EXTRACT_DIR / member.filename).resolve()
        if destination != extract_root and extract_root not in destination.parents:
            unsafe_members.append(member.filename)
    assert not unsafe_members, f"Unsafe paths in ZIP: {unsafe_members[:5]}"
    archive.extractall(EXTRACT_DIR)

# Accept a ZIP with or without one enclosing top-level folder. Training is driven by
# importing the codebase, so the markers are the modules this notebook imports.
repo_candidates = sorted(
    {
        main_path.parent.resolve()
        for main_path in EXTRACT_DIR.rglob("main.py")
        if (main_path.parent / "distiller.py").is_file()
        and (main_path.parent / "config" / "heatgeo_config.py").is_file()
    },
    key=lambda path: (len(path.parts), str(path)),
)
assert repo_candidates, (
    "The ZIP was extracted, but no HeatGeo repo containing main.py, distiller.py "
    "and config/heatgeo_config.py was found."
)
PROJECT_DIR = repo_candidates[0]
if len(repo_candidates) > 1:
    print(f"Multiple repo roots found; using the shallowest: {PROJECT_DIR}")

# All paths used by the remaining cells are finalized here.
TRAIN_DATA = PROJECT_DIR / "data" / "train_set" / "merged_3_data_5k_each.csv"
HEATGEO_CACHE_DIR = PROJECT_DIR / "cache" / "heatgeo"
RUN_DIR = PROJECT_DIR / "models" / "heatgeo" / "qwen3_4b_to_bert_base"
METRICS_PATH = RUN_DIR / "metrics.jsonl"

# Durable outputs live in Drive so they survive the Colab runtime: the training log,
# the per-epoch weights and the result tables.
DRIVE_RUN_DIR = DRIVE_DIR / "runs" / "qwen3_4b_to_bert_base"
LOG_PATH = DRIVE_RUN_DIR / "train.log"
TEST_BY_EPOCH_CSV = DRIVE_RUN_DIR / "test_by_epoch.csv"
FINAL_TEST_CSV = DRIVE_RUN_DIR / "final_test_results.csv"
WEIGHTS_DIR = DRIVE_DIR / "weights" / "qwen3_4b_to_bert_base"

assert (PROJECT_DIR / "requirements.txt").is_file(), "requirements.txt is missing"
assert TRAIN_DATA.is_file(), f"Training data not found: {TRAIN_DATA}"
# Evaluation reads the test split only, so no val_set file is required here.
required_split_files = {
    "train_set": {
        "merged_3_data_5k_each.csv", "banking77_train.csv",
        "emotion_train.csv", "tweet_train.csv",
    },
    "test_set": {
        "banking77_test.csv", "emotion_test.csv", "tweet_test.csv",
        "mrpc_test.csv", "scitail_test.csv", "wic_test.csv",
        "sick_test.csv", "sts12_test.csv", "stsb_test.csv",
    },
}
for split_dir, filenames in required_split_files.items():
    split_path = PROJECT_DIR / "data" / split_dir
    assert split_path.is_dir(), f"Data split directory not found: {split_path}"
    missing = [name for name in filenames if not (split_path / name).is_file()]
    assert not missing, f"Missing files in {split_path}: {missing}"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Resolved Colab paths:")
for name, path in {
    "archive": ARCHIVE_PATH,
    "extract": EXTRACT_DIR,
    "project": PROJECT_DIR,
    "train_data": TRAIN_DATA,
    "run_output": RUN_DIR,
    "drive_log": LOG_PATH,
    "drive_tables": DRIVE_RUN_DIR,
    "drive_weights": WEIGHTS_DIR,
}.items():
    print(f"  {name:12s}: {path}")
print(f"  {'student':12s}: {STUDENT_MODEL_NAME}")
print(f"  {'teacher':12s}: {TEACHER_MODEL_NAME}")

In [ ]:
# 3. Run from the repo root so the relative paths inside the codebase resolve, and
# make the repo importable: training runs inside this kernel, not in a subprocess.
import sys

%cd $PROJECT_DIR

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f"sys.path[0]: {sys.path[0]}")

In [ ]:
# 4. Install the project requirements into this Colab kernel.
# %pip streams its output straight into this cell.
%pip install -r requirements.txt

In [ ]:
# 5. Fail early if the extracted codebase is not configured for HeatGeo.
from config.heatgeo_config import HeatGeoConfig

cfg = HeatGeoConfig()
assert cfg.distill_method == "heatgeo", cfg.distill_method
assert cfg.eval_every == 1, f"Expected eval_every=1, got {cfg.eval_every}"
assert cfg.student_model_name == STUDENT_MODEL_NAME, cfg.student_model_name
assert cfg.teacher_model_name == TEACHER_MODEL_NAME, cfg.teacher_model_name
assert cfg.train_data_path == "data/train_set/merged_3_data_5k_each.csv"
print(
    "HeatGeo config verified: "
    f"method={cfg.distill_method}, eval_every={cfg.eval_every}, "
    f"student={cfg.student_model_name}, teacher={cfg.teacher_model_name}"
)
print("[READY] HeatGeo configuration is ready.")
print(
    "If pip asked for a restart, use Runtime > Restart session and re-run from cell 3."
)

In [ ]:
# 6. Report the accelerator that KnowledgeDistiller will use in this kernel.
import torch

cuda_count = torch.cuda.device_count()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
if cuda_count >= 2:
    selected = "student=cuda:0, teacher=cuda:1"
elif torch.cuda.is_available():
    selected = "student=cuda:0, teacher=cuda:0"
elif mps_available:
    selected = "student=mps, teacher=mps"
else:
    selected = "student=cpu, teacher=cpu"

print(f"Python environment: {sys.executable}")
print(f"PyTorch version: {torch.__version__} (CUDA build {torch.version.cuda})")
print(f"CUDA available: {torch.cuda.is_available()} (device count={cuda_count})")
for index in range(cuda_count):
    properties = torch.cuda.get_device_properties(index)
    print(
        f"  cuda:{index}: {properties.name} "
        f"({properties.total_memory / (1024 ** 3):.1f} GiB)"
    )
print(f"MPS available: {mps_available}")
print(f"KnowledgeDistiller will use: {selected}")
if torch.cuda.is_available():
    print("GPU STATUS: READY - HeatGeo training will use CUDA.")
else:
    print("GPU STATUS: NOT USING CUDA - enable a Colab GPU runtime before training.")

In [ ]:
# 7. Remove only stale HeatGeo cache and this reproduction run's output.
import shutil

for path in (HEATGEO_CACHE_DIR, RUN_DIR):
    resolved = path.resolve()
    assert PROJECT_DIR.resolve() in resolved.parents, f"Unsafe cleanup target: {resolved}"
    if path.is_symlink():
        path.unlink()
        print(f"Removed stale symlink: {path}")
    elif resolved.exists():
        shutil.rmtree(resolved)
        print(f"Removed stale artifacts: {resolved}")

RUN_DIR.mkdir(parents=True, exist_ok=False)
print(f"Clean run output: {RUN_DIR}")

In [ ]:
# 8. Train HeatGeo inside this kernel. Every line is printed in this cell and
# mirrored into the Drive log. Evaluation reads data/test_set/* after every epoch and
# once more at the end of training; data/val_set/* is never read.
import io
import os
from contextlib import redirect_stderr, redirect_stdout

import distiller as distiller_module
from distiller import KnowledgeDistiller
from src.evaluation.evaluation_automodel import (
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)

# KnowledgeDistiller.evaluate() resolves its per-epoch task lists from these module
# globals, so rebinding them points the per-epoch pass at the test files. Pair
# thresholds are refit on the test pairs instead of being carried over from val_set.
distiller_module.eval_cls_tasks = test_cls_tasks
distiller_module.eval_pair_tasks = test_pair_tasks
distiller_module.eval_sts_tasks = test_sts_tasks
for task_paths in (test_cls_tasks, test_pair_tasks, test_sts_tasks):
    for entry in task_paths:
        for path in (entry if isinstance(entry, tuple) else (entry,)):
            assert "val_set" not in path, f"Evaluation must not read val_set: {path}"


class TeeStream(io.TextIOBase):
    """Write to the notebook cell and to the Drive log, collapsing tqdm redraws."""

    def __init__(self, stream, handle):
        self.stream = stream
        self.handle = handle
        self._pending = ""

    def write(self, text):
        self.stream.write(text)
        self.stream.flush()
        self._pending += text
        wrote_line = False
        while "\n" in self._pending:
            line, self._pending = self._pending.split("\n", 1)
            # An in-place progress bar only needs its last redraw in the log file.
            self.handle.write(line.rsplit("\r", 1)[-1] + "\n")
            wrote_line = True
        self._pending = self._pending.rsplit("\r", 1)[-1]
        if wrote_line:
            self.handle.flush()
        return len(text)

    def flush(self):
        self.stream.flush()
        self.handle.flush()

    def writable(self):
        return True

    def isatty(self):
        return False


class TestSplitDistiller(KnowledgeDistiller):
    """KnowledgeDistiller whose tables are labelled with the split they read."""

    def print_evaluation_table(self, split, results):
        # Every pass reads data/test_set/*, so render with the test naming and only
        # relabel the heading of the per-epoch pass.
        rendered = io.StringIO()
        with redirect_stdout(rendered):
            super().print_evaluation_table("test", results)
        text = rendered.getvalue()
        if split != "test":
            text = text.replace(
                "FINAL TEST", f"TEST - EPOCH {self.current_epoch + 1}", 1
            )
        print(text, end="")


EPOCHS = 5
BATCH_SIZE = 16

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["WANDB_MODE"] = "disabled"

config = HeatGeoConfig()
config.train_data_path = str(TRAIN_DATA)
config.student_model_name = STUDENT_MODEL_NAME
config.teacher_model_name = TEACHER_MODEL_NAME
config.batch_size = BATCH_SIZE
config.epochs = EPOCHS
config.learning_rate = 2e-5
config.max_length = 256
config.save_dir = str(RUN_DIR)
config.weights_dir = str(WEIGHTS_DIR)
config.eval_every = 1
config.use_wandb = False

print("\n" + "=" * 80)
print("HeatGeo training configuration")
print("=" * 80)
print(f"Student model : {config.student_model_name}")
print(f"Teacher model : {config.teacher_model_name}")
print(f"Training data : {config.train_data_path}")
print(f"Batch size    : {config.batch_size}")
print(f"Epochs        : {config.epochs}")
print(f"Learning rate : {config.learning_rate}")
print(f"Evaluation    : data/test_set/* after every epoch and at the end")
print(f"Output        : {RUN_DIR}")
print(f"Drive log     : {LOG_PATH}")
print(f"Drive weights : {WEIGHTS_DIR}")
print("=" * 80)
print("Training output streams below and into the Drive log.\n")

cell_stdout = sys.stdout
with LOG_PATH.open("w", encoding="utf-8") as log_handle:
    tee = TeeStream(cell_stdout, log_handle)
    # stderr is merged into the same sink so tqdm bars land in the cell and the log.
    with redirect_stdout(tee), redirect_stderr(tee):
        print("=" * 70)
        print(f"Configuration for {config.distill_method.upper()} method:")
        print("=" * 70)
        for key, value in config.to_dict().items():
            print(f"  {key:25s} : {value}")
        print("=" * 70 + "\n")
        distiller = TestSplitDistiller(config)
        distiller.train()

print("\n[OK] HeatGeo training finished")
assert METRICS_PATH.is_file(), f"Training finished without metrics: {METRICS_PATH}"
print(f"[OK] Metrics file found: {METRICS_PATH}")
print(f"[OK] Full training log in Drive: {LOG_PATH}")
expected_weight_files = [
    WEIGHTS_DIR / f"student_epoch_{epoch}.pt" for epoch in range(1, EPOCHS + 1)
]
missing_weights = [
    path for path in expected_weight_files
    if not path.is_file() or path.stat().st_size == 0
]
assert not missing_weights, f"Missing or empty Drive weights: {missing_weights}"
print(f"[OK] Verified {len(expected_weight_files)} epoch weight files in Drive")

In [ ]:
# 9. Build the per-epoch test table and the final test table.
# Both come from data/test_set/*: the trainer writes each epoch's evaluation under the
# "validation" key of metrics.jsonl, and the end-of-training pass under "test".
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

METRIC_COLUMNS = [
    "accuracy",
    "f1",
    "precision",
    "recall",
    "average_precision",
    "spearman",
    "best_threshold",
]

def benchmark_name(path):
    name = Path(path).stem
    return name[:-len("_test")] if name.endswith("_test") else name

def result_rows(payload, epoch=None):
    rows = []
    for family in ("classification", "pair", "sts"):
        for path, result in payload.get(family, {}).items():
            row = {
                "family": family,
                "benchmark": benchmark_name(path),
            }
            if epoch is not None:
                row["epoch"] = int(epoch)
            if family == "sts":
                row["spearman"] = float(result)
            else:
                row.update(
                    {
                        metric: float(value)
                        for metric, value in result.items()
                        if metric in METRIC_COLUMNS
                    }
                )
            rows.append(row)
    return rows

def with_family_means(detail, group_columns, metric_columns):
    means = (
        detail.groupby(group_columns, as_index=False)[metric_columns]
        .mean()
        .assign(benchmark="MEAN")
    )
    table = pd.concat([detail, means], ignore_index=True)
    family_order = {"classification": 0, "pair": 1, "sts": 2}
    table["_family_order"] = table["family"].map(family_order)
    table["_mean_order"] = table["benchmark"].eq("MEAN")
    sort_columns = [
        column for column in group_columns if column != "family"
    ] + ["_family_order", "_mean_order", "benchmark"]
    return (
        table.sort_values(sort_columns)
        .drop(columns=["_family_order", "_mean_order"])
        .reset_index(drop=True)
    )

with METRICS_PATH.open(encoding="utf-8") as handle:
    records = [json.loads(line) for line in handle if line.strip()]

epoch_rows = []
for record in records:
    payload = record.get("validation")
    epoch = record.get("train", {}).get("epoch")
    if payload and epoch is not None:
        epoch_rows.extend(result_rows(payload, epoch))
assert epoch_rows, f"No per-epoch evaluation records found in {METRICS_PATH}"

epoch_detail = pd.DataFrame(epoch_rows).reindex(
    columns=["epoch", "family", "benchmark", *METRIC_COLUMNS]
)
test_by_epoch = with_family_means(epoch_detail, ["epoch", "family"], METRIC_COLUMNS)

test_payloads = [record["test"] for record in records if record.get("test")]
assert len(test_payloads) == 1, f"Expected one final test record, got {len(test_payloads)}"
test_detail = pd.DataFrame(result_rows(test_payloads[0])).reindex(
    columns=["family", "benchmark", *METRIC_COLUMNS]
)
final_test_results = with_family_means(test_detail, ["family"], METRIC_COLUMNS)

test_by_epoch.to_csv(TEST_BY_EPOCH_CSV, index=False)
final_test_results.to_csv(FINAL_TEST_CSV, index=False)
print("TEST RESULTS BY EPOCH")
display(test_by_epoch.style.format(precision=4, na_rep="—"))
print("FINAL TEST RESULTS")
display(final_test_results.style.format(precision=4, na_rep="—"))
print(f"Saved per-epoch test table: {TEST_BY_EPOCH_CSV}")
print(f"Saved final test table: {FINAL_TEST_CSV}")